# Questão 3 - Load Schemas with Datasets

In [34]:
#Premissas obrigatórias

#* Realize o carregamento de todos os CSVs.
#* Utilize obrigatoriamente Python 3.
#* Utilize qualquer biblioteca necessária (nativa ou externa) para conexão e carregamento dos dados.
#* Não faça tratamentos como: Remoção de nulos ou correção de caracteres especiais

In [35]:
#Escreva um script python para realizar o carregamento de todos os arquivos CSV respeitando o schema criado na questão anterior. 

In [36]:
#Importando bibliotecas necessárias
import csv
import os
import psycopg2
import json

In [37]:
#Definindo o diretório onde os arquivos CSVs estão localizados
DATASET_DIR = "Dataset"

#Banco de dados associado ao PostgreSQL (Configurado para conexão com localhost)
with open("Config/db_config.json", "r", encoding="utf-8") as file:
    config = json.load(file)

#Arquivo de configuração do banco de dados PostgreSQL não será carregado no GitHub por questões de segurança
#ip route | grep default > para verificar qual é o IP do host
DB_CONFIG = {
    "host": config["host"],
    "port": config["port"],
    "database": config["database"],
    "user": config["user"],
    "password": config["password"]
}

In [38]:
#Funções para tratar nomes de tabelas e colunas para o PostgreSQL sendo as mesmas usadas em Schemas.ipynb
def created_table_name(filename):
    """ Utiliza o nome do arquivo CSV como nome da tabela SQL."""
    table_name = os.path.splitext(filename)[0]

    return table_name.lower()

def created_column_name(column):
    """ Normaliza o nome das colunas para PostgreSQL."""

    #Tratamento de possiveis espaços e caracteres especiais no nome da coluna
    column = column.strip()
    column = column.replace(" ", "_")
    column = column.replace("-", "_")

    return column.lower()

In [39]:
#Função para inserir CSVs no PostgreSQL
def load_csv_to_postgres(connection, filepath):
    """ Insere os dados de um arquivo CSV em sua respectiva tabela PostgreSQL presente no arquivo schemas.sql. """

    filename = os.path.basename(filepath)
    table_name = created_table_name(filename)

    print(f"Carregando: {filename}")

    with open(
        filepath,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as csvfile:
        reader = csv.reader(csvfile)
        header = next(reader)
        columns = [
            created_column_name(column)
            for column in header
        ]
        column_list = ", ".join(
            f'"{column}"'
            for column in columns
        )
        placeholders = ", ".join(
            ["%s"] * len(columns)
        )

        #Ação de INSERT no PostgreSQL (SQL)
        insert_query = f"""
            INSERT INTO "{table_name}"
            ({column_list})
            VALUES ({placeholders})
        """

        cur = connection.cursor()

        for row in reader:
            cur.execute(
                insert_query,
                row
            )

        connection.commit()
        cur.close()

    print(f"  {filename} inserido com sucesso!")

In [40]:
#Função para chamar todos os arquivos CSVs e a partir da função load_csv_to_postgres inserir no PostgreSQL
def load_all_csvs():
    """ Carrega todos os arquivos CSV do diretório DATASET_DIR para o banco de dados PostgreSQL. """

    connection = psycopg2.connect(**DB_CONFIG)

    #Busca apenas arquivos CSV no diretório DATASET_DIR e chama a função load_csv_to_postgres para cada arquivo encontrado
    try:
        files = sorted(os.listdir(DATASET_DIR))
        for filename in files:
            if not filename.lower().endswith(".csv"):
                continue
            filepath = os.path.join(
                DATASET_DIR,
                filename
            )
            load_csv_to_postgres(
                connection,
                filepath
            )
    except Exception as error:
        connection.rollback()
        print(
            f"Erro durante o carregamento: {error}")
        raise

    finally:
        connection.close()

In [ ]:
#Chamando função principal para gerar o insert SQL a partir dos arquivos CSVs
load_all_csvs()